In [ ]:
 
# =====================================================================
# CELL 1 — Setup: copy YOLOX repo, install YOLOX
# =====================================================================
import os, shutil, subprocess, sys
 
print("STAGE 1/5: SETUP")
YOLOX_SRC = '/kaggle/input/datasets/punnamrahul/lcam-v5-finetune/kaggle_v5_upload/YOLOX'
YOLOX_DIR = '/kaggle/working/YOLOX'
 
if os.path.exists(YOLOX_DIR):
    shutil.rmtree(YOLOX_DIR)
shutil.copytree(YOLOX_SRC, YOLOX_DIR)
print("YOLOX copied.")
 
os.chdir(YOLOX_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'loguru', 'thop', 'tabulate', 'pycocotools'], check=False)
# install YOLOX in editable mode
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=False)
print("Deps installed. STAGE 1 done.")
 

In [ ]:
import os
p = '/kaggle/working/YOLOX_outputs/lcam_yolox_wildfire/best_ckpt.pth'
print("Exists:", os.path.exists(p))
if os.path.exists(p):
    print("Size:", os.path.getsize(p), "bytes")

In [ ]:
import os, glob

print("=== Checkpoints (.pth) ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pth'):
            print(" ", os.path.join(root, f))

print("\n=== YOLOX code ===")
for root, dirs, files in os.walk('/kaggle/input'):
    if root.endswith('YOLOX') and 'yolox' in dirs:
        print(" ", root)

print("\n=== D-Fire test images ===")
for root, dirs, files in os.walk('/kaggle/input'):
    if root.endswith('test/images'):
        n = len(glob.glob(f'{root}/*.jpg'))
        print(f"  {root}  ({n} images)")

print("\n=== Exp file ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'lcam_yolox_wildfire' in f and f.endswith('.py'):
            print(" ", os.path.join(root, f))

In [ ]:
import os, sys, shutil, glob

# Copy YOLOX to working dir (so we can use/patch it)
YSRC = '/kaggle/input/datasets/punnamrahul/lcam-v5-finetune/kaggle_v5_upload/YOLOX'
YDST = '/kaggle/working/YOLOX'
if os.path.exists(YDST): shutil.rmtree(YDST)
shutil.copytree(YSRC, YDST)

# Check if lcam.py is v5
src = open(f'{YDST}/yolox/models/lcam.py').read()
print("lcam.py is v5:", 'StagedGlobalAvgPool' in src and 'Hardsigmoid' in src)

# Install deps + fix mlflow (same fixes as training)
os.system('pip install loguru thop tabulate pycocotools -q')
# neutralize mlflow in trainer/utils
for path, reps in [
    (f'{YDST}/yolox/utils/__init__.py', [('from .mlflow_logger import MlflowLogger', '# from .mlflow_logger import MlflowLogger')]),
    (f'{YDST}/yolox/core/trainer.py', [('MlflowLogger,', ''), ('MlflowLogger', 'object')]),
]:
    if os.path.exists(path):
        c = open(path).read()
        for a,b in reps: c = c.replace(a,b)
        open(path,'w').write(c)
print("Setup done.")

In [ ]:
import sys, os, torch, cv2, numpy as np, glob
sys.path.insert(0, '/kaggle/working/YOLOX')
# clear cached yolox modules
for m in list(sys.modules.keys()):
    if m.startswith('yolox'): del sys.modules[m]
from yolox.exp import get_exp
from yolox.utils import postprocess
import matplotlib.pyplot as plt

# --- Build v5 model (same config as training) ---
exp = get_exp(exp_file=None, exp_name='yolox-s')
exp.num_classes = 2
exp.act = 'lrelu'
model = exp.get_model()

CKPT = '/kaggle/input/models/punnamrahul/finetune-hardsigmoid/pytorch/default/1/best_ckpt (1).pth'
ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)   # <-- added weights_only=False
sd = ckpt['model'] if 'model' in ckpt else ckpt
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f"Loaded. Missing: {len(missing)}, Unexpected: {len(unexpected)}")
print("Best AP from training:", ckpt.get('best_ap', 'n/a'))
model = model.cuda().eval()

CLASSES = ['smoke', 'fire']
TEST_DIR = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images'
img_files = sorted(glob.glob(f'{TEST_DIR}/*.jpg'))

def detect(img_path, conf_thre=0.3):
    img0 = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h0, w0 = img0.shape[:2]
    r = min(640/h0, 640/w0)
    rimg = cv2.resize(img0, (int(w0*r), int(h0*r)))
    canvas = np.full((640,640,3), 114, np.uint8)
    canvas[:rimg.shape[0], :rimg.shape[1]] = rimg
    x = torch.from_numpy(canvas.transpose(2,0,1)).float().unsqueeze(0).cuda()
    with torch.no_grad():
        out = postprocess(model(x), 2, conf_thre, 0.45)[0]
    vis = img0.copy()
    n = 0
    if out is not None:
        out = out.cpu().numpy()
        n = len(out)
        for det in out:
            x1,y1,x2,y2 = (det[:4]/r).astype(int)
            score = det[4]*det[5]; cls = int(det[6])
            color = (255,0,0) if cls==1 else (255,150,0)
            cv2.rectangle(vis,(x1,y1),(x2,y2),color,3)
            cv2.putText(vis,f'{CLASSES[cls]} {score:.2f}',(x1,max(y1-8,15)),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,color,2)
    return vis, n

fig, axes = plt.subplots(2,3, figsize=(18,12))
for ax, idx in zip(axes.flat, [10, 200, 800, 1500, 2500, 4000]):
    vis, n = detect(img_files[idx])
    ax.imshow(vis); ax.axis('off')
    ax.set_title(f'{os.path.basename(img_files[idx])} — {n} boxes')
plt.tight_layout(); plt.savefig('/kaggle/working/v5_detections.png', dpi=70); plt.show()
print("Done — check the images above.")